### Data Ingestion


### document dataStructure


In [5]:
from langchain_core.documents import Document

In [6]:
doc=Document(
    page_content="this is the main text content i am using to create RAG",
    metadata={
        "source":"example.txt",
        "pages":1,
        "author":"Maneth Kaveen",
        "date_Created":"2026-11-15"
    }
)

In [7]:
doc

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'Maneth Kaveen', 'date_Created': '2026-11-15'}, page_content='this is the main text content i am using to create RAG')

In [8]:
### create a simple txt 

import os
os.makedirs("../data/text_files",exist_ok=True)

In [9]:
sample_texts={
    "../data/text_files/python_intro.txt":"""Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
    
    "../data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems"""
}

for filepath,content in sample_texts.items():
    with open(filepath,"w",encoding="utf-8") as f:
        f.write(content)

print("sample text file created!")

sample text file created!


In [10]:
### TextLoader

from langchain_community.document_loaders import TextLoader

loader=TextLoader("../data/text_files/python_intro.txt",encoding="utf-8")
document=loader.load()
print(document)

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')]


C:\Users\User\AppData\Local\Temp\ipykernel_31164\2416650908.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [11]:
### Document Loader
from langchain_community.document_loaders import DirectoryLoader

## load all the text files from the directory
dir_loader=DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt", ## Pattern to match files  
    loader_cls= TextLoader, ##loader class to use
    loader_kwargs={'encoding': 'utf-8'},
    show_progress=False

)

documents=dir_loader.load()
documents

[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems'),
 Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogr

In [12]:
# Creating Data Chunks 

from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """
    Split documents into smaller chunks for better RAG performance.
    
    Parameters:
    - chunk_size: Maximum characters per chunk (adjust based on your LLM)
    - chunk_overlap: Characters to overlap between chunks (preserves context)
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, # Each chunk: ~1000 characters
        chunk_overlap=chunk_overlap, # 200 chars overlap for context
        length_function=len, # How to measure length
        separators=["\n\n", "\n", " ", ""] # Split hierarchy
    )
    # Actually split the documents
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show what a chunk looks like
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [13]:
chunks=split_documents(documents)
chunks

Split 2 documents into 2 chunks

Example chunk:
Content: Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing...
Metadata: {'source': '..\\data\\text_files\\machine_learning.txt'}


[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems'),
 Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogr

In [14]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

## load all the text files from the directory
dir_loader=DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf", ## Pattern to match files  
    loader_cls= PyMuPDFLoader, ##loader class to use
    show_progress=False

)

pdf_documents=dir_loader.load()
pdf_documents

[Document(metadata={'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2026-02-03T06:18:59+00:00', 'source': '..\\data\\pdf\\20240753.pdf', 'file_path': '..\\data\\pdf\\20240753.pdf', 'total_pages': 4, 'format': 'PDF 1.4', 'title': 'Placement Agreement UOW.pdf', 'author': 'Induwara Dilshan', 'subject': '', 'keywords': 'DAHAPZ5tlrQ,BAF-LPitw3A,0', 'moddate': '2026-02-03T06:18:58+00:00', 'trapped': '', 'modDate': "D:20260203061858+00'00'", 'creationDate': "D:20260203061859+00'00'", 'page': 0}, page_content='1. Student Responsibilities \nI. \nBefore the Placement\n Industrial Placement Agreement (UOW) \nThis Agreement is required for any Industrial Placement undertaken as part of a course \nwithin the University of Westminster (UOW). \nThe Student must sign this agreement to initiate this approved work Industrial Placement \nAgreement. This agreement is required where the work placement is directly related to the \ncourse of study being undertaken by the Student. \nThe purpose of t

In [15]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity 

c:\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [16]:
class EmbeddingManager:
    def __init__(self,model_name:str="all-MiniLM-L6-v2"):
        """Initialize the embedding manager 
        Args:
            model name: Hugging face model name for sentence embeddings
            """
        self.model_name =model_name
        self.model=None
        self._load_model()

    def _load_model(self):
        """Load the sentenceTransformer model"""
        try:
            print(f"Loading embedding model:{self.model_name}")
            self.model =SentenceTransformer(self.model_name)
            print(f"model loaded successfully. embedding dimension:{self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}:{e}")
            raise 
    def generate_embeddings(self,texts:List[str])->np.ndarray:
        """
        Generate embeddingfs for a list of texts
        
        Args:
            texts:List of text strings to embed
            
        returns:
            numpy array of embeddings with shape(len(texts),embedding_dim)
        """

        if not self.model:
            raise ValueError("model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts,show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

### initialize the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model:all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 942.87it/s]


model loaded successfully. embedding dimension:384


C:\Users\User\AppData\Local\Temp\ipykernel_31164\2530458273.py:16: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"model loaded successfully. embedding dimension:{self.model.get_sentence_embedding_dimension()}")


In [17]:
import os
### vector store
class VectorStore:
    """Manages document embeddings in a chromaDB vector store"""

    def __init__(self,collection_name:str="pdf_documents",persist_directory:str= "../data/vector_store"):
        """Initialize the vector store
        
        Args:
            collection_name:Name of the chromaDB collection
            persist_directory:Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory=persist_directory
        self.client=None
        self.collection=None
        self._initialize_store()
    
    def _initialize_store(self):
        """Initialize chromaDB client and collection"""

        try:
            # create persistent chromaDB client
            os.makedirs(self.persist_directory,exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_directory)

            #Get or create collection
            self.collection=self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description":"PDF document embeddings for RAG"}
            )
            print(f"Vecotr store initialized. Collection :{self.collection_name}")
            print(f"existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store:{e}")
            raise

    def add_documents(self,documents:List[Any],embeddings:np.ndarray):
        """Add documents and their embeddings to the vector store
        Args:
            documents:list of langchain documents
            embeddings: Corresponding embeddings for the documents 
        """
        if(len(documents)!= len(embeddings)):
            raise ValueError("Number of documents must match number of embeddings")
        print(f"Adding {len(documents)} documents to vector store")

        #prepare data for chromaDB
        ids=[]
        metadatas=[]
        documents_text=[]
        embeddings_list=[]

        for i,(doc,embedding) in enumerate(zip(documents,embeddings)):
            # generate unique ID
            doc_id=f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # prepare metadata
            metadata=dict(doc.metadata)
            metadata['doc index'] = i
            metadata['content_length']=len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"successfully added {len(documents)} documents to vector store")
            print(f"total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"error adding documents to vector store {e}")
            raise
vectorestore=VectorStore()
vectorestore

Vecotr store initialized. Collection :pdf_documents
existing documents in collection: 6


In [18]:
chunks 

[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems'),
 Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogr

In [19]:
### convert the text to embeddings
texts=[doc.page_content for doc in chunks]
texts

### generate the embeddings

embeddings=embedding_manager.generate_embeddings(texts)

### store in the vector DB
vectorestore.add_documents(chunks,embeddings)


Generating embeddings for 2 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  8.41it/s]

Generated embeddings with shape: (2, 384)
Adding 2 documents to vector store
successfully added 2 documents to vector store
total documents in collection: 8


In [20]:
from sqlalchemy import exc
from importlib import metadata
### Retriever Pipeline from VectorStore


class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    def __init__(self,vector_store:VectorStore,embedding_manager:EmbeddingManager):
        """Initialize the retriever
        
        Args:
            Vecotr_store:Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vectore_store = vector_store
        self.embedding_manager = embedding_manager
    
    def retrieve(self,query:str,top_k:int =5,score_threshold:float=0.0)-> List[dict[str,Any]]:
        """
        Retrieve relevant documents for a query
        Args: 
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictationaries containing retrieved documents and metadata
        """

        print(f"Retrieve documents for query: '{query}'")
        print(f"top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector store
        try:
            results = self.vectore_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results = top_k
            )

            # Process results
            retrieved_docs=[]
            if results['documents'] and results['documents'][0]:
                documents=results['documents'][0]
                metadatas = results["metadatas"][0]
                distances = results['distances'][0]
            
            for i,(doc_id,document,metadata,distance) in enumerate(zip(results['ids'][0],documents,metadatas,distances)):
                # Convert distances to similarity score (ChromaDB uses cosine distamce)
                similarity_score = 1-distance

                if similarity_score>= score_threshold:
                    retrieved_docs.append({
                        'id':doc_id,
                        'content':document,
                        'metadata': metadata,
                        'similarity_score': similarity_score,
                        'distance':distance,
                        "rank":i+1
                    })
                    print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
                else:
                    print("No documents found")
                return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval : {e}")
            return []
rag_retriever = RAGRetriever(vectorestore,embedding_manager)
                    

In [21]:
rag_retriever

In [22]:
rag_retriever.retrieve("Hey what are the types of Machine Learning")

Retrieve documents for query: 'Hey what are the types of Machine Learning'
top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 85.87it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


[{'id': 'doc_93a15ac3_0',
  'content': 'Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems',
  'metadata': {'content_length': 568,
   'source': '..\\data\\text_files\\machine_learning.txt',
   'doc index': 0},
  'similarity_score': 0.5499829351902008,
  'distance': 0.4500170648097992,
  'rank': 1}]

### intergration vectordb context pipleline with LLM output

In [ ]:
### simple rag pipeline 
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
load_dotenv()

### initialize the gemini LLM
gemini_api_key=os.getenv('GOOGLE_API_KEY')

llm = ChatGoogleGenerativeAI(
    model='gemini-3.8-flash',
    google_api_key=gemini_api_key,
    temperature=0.1,
    max_output_tokens=1024,
)


In [26]:
print(os.getenv("GOOGLE_API_KEY"))

AQ.Ab8RN6I-QgR_-1_VL92_J15B2IGBJNK0ltbOv4BsWb_7U6WcwA


In [24]:
### 2. Simple RAG function : retrieve context + generate responses 

def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results = retriever.retrieve(query,top_k=top_k)
    context = "\n\n".join([doc["content"] for doc in results]) if results else ""

    if not context:
        return "No relevant context found to answer the question."

    ## generate the answer using gemini LLM
    prompt= f"""Use the following context to answer the question concisely.

        Context:
        {context}

        Question: {query}

        Answer:"""
    # 3. Generate answer with Gemini
    response = llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [25]:
answer= rag_simple("Hey what are the types of Machine Learning",rag_retriever,llm)

Retrieve documents for query: 'Hey what are the types of Machine Learning'
top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 87.81it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


In [27]:
answer

[{'type': 'text',
  'text': 'The types of machine learning are:\n1. Supervised Learning (learning with labeled data)\n2. Unsupervised Learning (finding patterns in unlabeled data)\n3. Reinforcement Learning (learning through rewards and penalties)',
  'extras': {'signature': 'EroGCrcGAWkUfROAjQD/T99j9MU4NzCcaUqYpyYIbMfTT81Ggv59HN64X1tTHqGIVinvhO7NYwvugS2n0gRWphdXAaPbMcpqY6+h8sGpDFSpKmqFx4G7H9PD0Ivf7IUfo4C6HPaW9yOcxVAJ7nhTw6CM5h9z3kZHW7YsAiRC6IORoYmZy5/4uLKhqTGIpRU5ZEr2gCSep1jG4Hb2Al9T6iQa9n/65sOr1MRmmaGEQG8P+4p6LMuyq5VuYUiWriZ/hLFvIDQ54Vb+1tMpeG/j3uFi1F+SK3PWTwAUi1M6+yLClrA9Y9DSeCd/JVT6O2RD2TWwZGbHGCeCMf4rKhpt7SizAEUs29DXo3BeuIXkMQCR86OFKGSJsNvKIQC/mPx22R60gXowLBMPmmVIKXqDrww7tVITmXT2CxjBDmY4/KcplqInK2EvmvmckyGTLddqJuLSfkcjY5uRAQAPDY8NwVEULbRX9ESw/YT7qJg34mY1ykm73Vi3uvbwWrW0iRIb7VHg7w2be/wmjfiBCCq0dLhdKeO2lrqr0SS9LWe82IlOGvDk1nxCVYc3sN8uiSalIOqzftNUuBZWozvumo8Sk+qq0TAFPkFerb90TiLrH9eIhlihiLdxGrejZ2KtSOSG8i39ufyp46wBMh0qmjX3AfHYVPCSmL0c5WXQ/TDiZuUGuZCVlTa5fUDSV7n1fslJa3CwKtpCjCnm0ed9pD9

In [ ]:
answer[0]['text']

'The types of machine learning are:\n1. Supervised Learning (learning with labeled data)\n2. Unsupervised Learning (finding patterns in unlabeled data)\n3. Reinforcement Learning (learning through rewards and penalties)'

: 

In [ ]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)

    if not results:
        return {
            "answer": "No relevant context found.",
            "sources": [],
            "confidence": 0.0,
            "context": ""
        }

    context = "\n\n".join([doc["content"] for doc in results])
    sources = [
        {
            "source": doc["metadata"].get("source_file", doc["metadata"].get("source", "unknown")),
            "page": doc["metadata"].get("page", "unknown"),
            "score": doc["similarity_score"],
            "preview": doc["content"][:300] + "..."
        }
        for doc in results
    ]
    confidence = max(doc["similarity_score"] for doc in results)

    prompt = f"""Use the following context to answer the question concisely.

Context:
{context}

Question: {query}

Answer:"""

    response = llm.invoke(prompt)

    output = {
        "answer": response.content,
        "sources": sources,
        "confidence": confidence,
    }
    if return_context:
        output["context"] = context

    return output

# Run the enhanced pipeline
result = rag_advanced("What is attention mechanism?", rag_retriever, llm, return_context=True)
print("Answer:", result["answer"])
print("Sources:", result["sources"])
print("Confidence:", result["confidence"])